# Extended Data Fig. 10g — NMP and LPM marker transcription factors

Author: Yusuf Ilker Yaman. Ported from two notebooks: `rebuild_combined_expression_matrix.ipynb` and
`expression_threshold_qc.ipynb`.

The panel is a heatmap of 24 transcription factors across pluripotent cysts, neuromesodermal progenitors
(NMP) and lateral plate mesoderm (LPM), three samples each. Values are log2(CPM + 1), z-scored per gene.
Before plotting, genes whose mean CPM is below 2 in all three conditions are removed and the table is
restricted to transcription factors from the Human Protein Atlas. The 24 genes shown are a named list,
not a statistical selection: the ANOVA and pairwise Welch's t-tests with Benjamini–Hochberg correction
below are exported for reference and nothing downstream reads them. CPM values are used exactly as
computed by the sequencing provider; nothing here re-normalizes them.

**Changes from the original notebooks**
1. Input is the GEO series GSE347564 CPM table. The first original notebook read three per-order
   matrices and joined them on gene annotation (inner join); here the nine columns are selected from
   the GEO table and genes with a missing value in any of them are dropped, which keeps the same genes.
   Column names change accordingly: `pSMAD_0h_rep1–3` → `pluripotent_sample_1–3` (the pluripotent group
   is the 0 h samples of the BMP4 time course), `NMP_rep1–3` → `NMP_1–3`, `LPM_rep1–3` → `LPM_1–3`.
2. Paths are relative to the repository, with a check that the input table exists; outputs go to `bulkseq/output/`. The transcription-factor list
   ships as `bulkseq/derived/protein_class_Transcription.tsv`.
3. Removed cells that the panel does not use: the pooled-CPM histogram and threshold table, and a top-30
   marker heatmap that is not in the paper.
4. Added the last cell, which writes the plotted z-scores to `bulkseq/output/ed10g_heatmap_zscores.tsv`.

In [ ]:
# If needed, run this once in Jupyter before continuing:
# %pip install numpy pandas matplotlib seaborn scipy

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, ttest_ind

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 140

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bulkseq" / "README.md").is_file())
CPM_TABLE = Path(os.environ.get(
    "BULK_CPM_TABLE", REPO / "data" / "GSE347564" / "GSE347564_bulk_RNAseq_cpm_all_samples.tsv.gz"))
OUT = REPO / "bulkseq" / "output"
OUT.mkdir(parents=True, exist_ok=True)

tf_list_path = REPO / "bulkseq" / "derived" / "protein_class_Transcription.tsv"

lowexpr_filtered_path = OUT / "combined_cpm_expression_matrix_filtered_mean2.tsv"
tf_filtered_matrix_path = OUT / "combined_cpm_expression_matrix_filtered_mean2_TFonly.tsv"
marker_results_path = OUT / "marker_gene_statistics_log2cpm.tsv"
pluripotent_markers_path = OUT / "pluripotent_candidate_markers_log2cpm.tsv"
lpm_markers_path = OUT / "lpm_candidate_markers_log2cpm.tsv"
nmp_markers_path = OUT / "nmp_candidate_markers_log2cpm.tsv"
pluripotent_negative_markers_path = OUT / "pluripotent_negative_markers_log2cpm.tsv"
lpm_negative_markers_path = OUT / "lpm_negative_markers_log2cpm.tsv"
nmp_negative_markers_path = OUT / "nmp_negative_markers_log2cpm.tsv"
requested_gene_heatmap_path = OUT / "requested_gene_list_heatmap.png"
requested_gene_heatmap_pdf_path = OUT / "requested_gene_list_heatmap.pdf"

sample_columns = [
    "pluripotent_sample_1", "pluripotent_sample_2", "pluripotent_sample_3",
    "LPM_1", "LPM_2", "LPM_3",
    "NMP_1", "NMP_2", "NMP_3",
]

sample_groups = {
    "Pluripotent": ["pluripotent_sample_1", "pluripotent_sample_2", "pluripotent_sample_3"],
    "LPM": ["LPM_1", "LPM_2", "LPM_3"],
    "NMP": ["NMP_1", "NMP_2", "NMP_3"],
}

requested_genes = [
    "TEAD4",
    "HESX1", "POU5F1", "NANOG", "SOX2", "MYC", "GBX2", "TBXT", "HIF1A", "BACH2", "NKX1-2", "SIX4", "SP8", "ONECUT2",
    "LEF1", "CDX2", "ZEB2", "SNAI2", "GATA4", "FOXF1", "HAND1", "HAND2", "PRRX1", "EN1",
]

requested_sample_order = [
    "pluripotent_sample_1", "pluripotent_sample_2", "pluripotent_sample_3",
    "NMP_1", "NMP_2", "NMP_3",
    "LPM_1", "LPM_2", "LPM_3",
]

column_colors_default = pd.Series(
    ["#1b9e77"] * 3 + ["#d95f02"] * 3 + ["#7570b3"] * 3,
    index=sample_columns,
)

column_colors_requested = pd.Series(
    ["#1b9e77"] * 3 + ["#7570b3"] * 3 + ["#d95f02"] * 3,
    index=requested_sample_order,
)

def benjamini_hochberg(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    result = np.full_like(pvalues, np.nan, dtype=float)
    valid = np.isfinite(pvalues)
    if not valid.any():
        return result
    valid_p = pvalues[valid]
    n = len(valid_p)
    order = np.argsort(valid_p)
    ranked = valid_p[order]
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    valid_result = np.empty_like(valid_p)
    valid_result[order] = adjusted
    result[valid] = valid_result
    return result

def row_zscore(row):
    std = row.std()
    if std == 0 or pd.isna(std):
        return row * 0
    return (row - row.mean()) / std

In [ ]:
annotation_columns = ["gene_id", "gene_name", "gene_biotype"]
sample_mapping = {
    "pSMAD_0h_rep1": "pluripotent_sample_1",
    "pSMAD_0h_rep2": "pluripotent_sample_2",
    "pSMAD_0h_rep3": "pluripotent_sample_3",
    "LPM_rep1": "LPM_1",
    "LPM_rep2": "LPM_2",
    "LPM_rep3": "LPM_3",
    "NMP_rep1": "NMP_1",
    "NMP_rep2": "NMP_2",
    "NMP_rep3": "NMP_3",
}

if not CPM_TABLE.is_file():
    raise FileNotFoundError(f"CPM table not found: {CPM_TABLE}. See bulkseq/README.md.")
geo = pd.read_csv(CPM_TABLE, sep="\t")
missing = [c for c in annotation_columns + list(sample_mapping) if c not in geo.columns]
if missing:
    raise ValueError(f"Matrix missing columns: {missing}")

combined_df = geo[annotation_columns + list(sample_mapping)].rename(columns=sample_mapping)
combined_df = combined_df.dropna(subset=list(sample_mapping.values()))

final_column_order = [
    "gene_id",
    "gene_name",
    "gene_biotype",
    "pluripotent_sample_1",
    "pluripotent_sample_2",
    "pluripotent_sample_3",
    "LPM_1",
    "LPM_2",
    "LPM_3",
    "NMP_1",
    "NMP_2",
    "NMP_3",
]

df = combined_df[final_column_order]
print(f"Number of genes: {len(df)}")

In [ ]:
condition_means = pd.DataFrame({
    "mean_pluripotent": df[sample_groups["Pluripotent"]].mean(axis=1),
    "mean_lpm": df[sample_groups["LPM"]].mean(axis=1),
    "mean_nmp": df[sample_groups["NMP"]].mean(axis=1),
})

lowexpr_keep_mask = ~(
    (condition_means["mean_pluripotent"] < 2)
    & (condition_means["mean_lpm"] < 2)
    & (condition_means["mean_nmp"] < 2)
)

lowexpr_filtered_df = df.loc[lowexpr_keep_mask].copy()
lowexpr_filtered_df.to_csv(lowexpr_filtered_path, sep="\t", index=False)

tf_table = pd.read_csv(tf_list_path, sep="\t")
tf_gene_names = set(tf_table["Gene"].dropna().astype(str).str.upper())
tf_ensembl_ids = set(tf_table["Ensembl"].dropna().astype(str).str.upper())

tf_keep_mask = (
    lowexpr_filtered_df["gene_name"].fillna("").astype(str).str.upper().isin(tf_gene_names)
    | lowexpr_filtered_df["gene_id"].fillna("").astype(str).str.upper().isin(tf_ensembl_ids)
)

filtered_df = lowexpr_filtered_df.loc[tf_keep_mask].copy()
filtered_df.to_csv(tf_filtered_matrix_path, sep="\t", index=False)

filter_summary = pd.Series({
    "genes_before_filtering": len(df),
    "genes_after_low_expression_filter": len(lowexpr_filtered_df),
    "genes_in_tf_list": len(filtered_df),
    "genes_removed_as_non_TF": int((~tf_keep_mask).sum()),
})

display(filter_summary)
display(filtered_df.head())

print(f"Saved low-expression filtered matrix to: {lowexpr_filtered_path}")
print(f"Saved TF-only filtered matrix to: {tf_filtered_matrix_path}")

In [ ]:
analysis_df = filtered_df.copy()
log2_df = analysis_df.copy()
log2_df[sample_columns] = np.log2(log2_df[sample_columns] + 1)

records = []
for _, row in log2_df.iterrows():
    pluri = row[sample_groups["Pluripotent"]].to_numpy(dtype=float)
    lpm = row[sample_groups["LPM"]].to_numpy(dtype=float)
    nmp = row[sample_groups["NMP"]].to_numpy(dtype=float)

    try:
        anova_p = f_oneway(pluri, lpm, nmp).pvalue
    except ValueError:
        anova_p = 1.0

    pluri_vs_lpm_p = ttest_ind(pluri, lpm, equal_var=False).pvalue
    pluri_vs_nmp_p = ttest_ind(pluri, nmp, equal_var=False).pvalue
    lpm_vs_nmp_p = ttest_ind(lpm, nmp, equal_var=False).pvalue

    mean_pluri = pluri.mean()
    mean_lpm = lpm.mean()
    mean_nmp = nmp.mean()

    records.append({
        "gene_id": row["gene_id"],
        "gene_name": row["gene_name"],
        "gene_biotype": row["gene_biotype"],
        "mean_log2cpm_pluripotent": mean_pluri,
        "mean_log2cpm_lpm": mean_lpm,
        "mean_log2cpm_nmp": mean_nmp,
        "pluri_minus_lpm": mean_pluri - mean_lpm,
        "pluri_minus_nmp": mean_pluri - mean_nmp,
        "lpm_minus_nmp": mean_lpm - mean_nmp,
        "anova_p": anova_p,
        "pluri_vs_lpm_p": pluri_vs_lpm_p,
        "pluri_vs_nmp_p": pluri_vs_nmp_p,
        "lpm_vs_nmp_p": lpm_vs_nmp_p,
    })

marker_results = pd.DataFrame(records)
marker_results["anova_fdr"] = benjamini_hochberg(marker_results["anova_p"])
marker_results["pluri_vs_lpm_fdr"] = benjamini_hochberg(marker_results["pluri_vs_lpm_p"])
marker_results["pluri_vs_nmp_fdr"] = benjamini_hochberg(marker_results["pluri_vs_nmp_p"])
marker_results["lpm_vs_nmp_fdr"] = benjamini_hochberg(marker_results["lpm_vs_nmp_p"])
marker_results.to_csv(marker_results_path, sep="\t", index=False)

pluripotent_markers = marker_results[(marker_results["mean_log2cpm_pluripotent"] > marker_results["mean_log2cpm_lpm"]) & (marker_results["mean_log2cpm_pluripotent"] > marker_results["mean_log2cpm_nmp"]) & (marker_results["anova_fdr"] < 0.05) & (marker_results["pluri_vs_lpm_fdr"] < 0.05) & (marker_results["pluri_vs_nmp_fdr"] < 0.05) & (marker_results["pluri_minus_lpm"] > 0) & (marker_results["pluri_minus_nmp"] > 0)].sort_values(["anova_fdr", "pluri_minus_lpm", "pluri_minus_nmp"], ascending=[True, False, False])
lpm_markers = marker_results[(marker_results["mean_log2cpm_lpm"] > marker_results["mean_log2cpm_pluripotent"]) & (marker_results["mean_log2cpm_lpm"] > marker_results["mean_log2cpm_nmp"]) & (marker_results["anova_fdr"] < 0.05) & (marker_results["pluri_vs_lpm_fdr"] < 0.05) & (marker_results["lpm_vs_nmp_fdr"] < 0.05) & (marker_results["pluri_minus_lpm"] < 0) & (marker_results["lpm_minus_nmp"] > 0)].sort_values(["anova_fdr", "mean_log2cpm_lpm"], ascending=[True, False])
nmp_markers = marker_results[(marker_results["mean_log2cpm_nmp"] > marker_results["mean_log2cpm_pluripotent"]) & (marker_results["mean_log2cpm_nmp"] > marker_results["mean_log2cpm_lpm"]) & (marker_results["anova_fdr"] < 0.05) & (marker_results["pluri_vs_nmp_fdr"] < 0.05) & (marker_results["lpm_vs_nmp_fdr"] < 0.05) & (marker_results["pluri_minus_nmp"] < 0) & (marker_results["lpm_minus_nmp"] < 0)].sort_values(["anova_fdr", "mean_log2cpm_nmp"], ascending=[True, False])

pluripotent_negative_markers = marker_results[(marker_results["mean_log2cpm_pluripotent"] < marker_results["mean_log2cpm_lpm"]) & (marker_results["mean_log2cpm_pluripotent"] < marker_results["mean_log2cpm_nmp"]) & (marker_results["anova_fdr"] < 0.05) & (marker_results["pluri_vs_lpm_fdr"] < 0.05) & (marker_results["pluri_vs_nmp_fdr"] < 0.05) & (marker_results["pluri_minus_lpm"] < 0) & (marker_results["pluri_minus_nmp"] < 0)].sort_values(["anova_fdr", "mean_log2cpm_pluripotent"], ascending=[True, True])
lpm_negative_markers = marker_results[(marker_results["mean_log2cpm_lpm"] < marker_results["mean_log2cpm_pluripotent"]) & (marker_results["mean_log2cpm_lpm"] < marker_results["mean_log2cpm_nmp"]) & (marker_results["anova_fdr"] < 0.05) & (marker_results["pluri_vs_lpm_fdr"] < 0.05) & (marker_results["lpm_vs_nmp_fdr"] < 0.05) & (marker_results["pluri_minus_lpm"] > 0) & (marker_results["lpm_minus_nmp"] < 0)].sort_values(["anova_fdr", "mean_log2cpm_lpm"], ascending=[True, True])
nmp_negative_markers = marker_results[(marker_results["mean_log2cpm_nmp"] < marker_results["mean_log2cpm_pluripotent"]) & (marker_results["mean_log2cpm_nmp"] < marker_results["mean_log2cpm_lpm"]) & (marker_results["anova_fdr"] < 0.05) & (marker_results["pluri_vs_nmp_fdr"] < 0.05) & (marker_results["lpm_vs_nmp_fdr"] < 0.05) & (marker_results["pluri_minus_nmp"] > 0) & (marker_results["lpm_minus_nmp"] > 0)].sort_values(["anova_fdr", "mean_log2cpm_nmp"], ascending=[True, True])

pluripotent_markers.to_csv(pluripotent_markers_path, sep="\t", index=False)
lpm_markers.to_csv(lpm_markers_path, sep="\t", index=False)
nmp_markers.to_csv(nmp_markers_path, sep="\t", index=False)
pluripotent_negative_markers.to_csv(pluripotent_negative_markers_path, sep="\t", index=False)
lpm_negative_markers.to_csv(lpm_negative_markers_path, sep="\t", index=False)
nmp_negative_markers.to_csv(nmp_negative_markers_path, sep="\t", index=False)

display(pd.Series({
    "positive_pluripotent": len(pluripotent_markers),
    "positive_lpm": len(lpm_markers),
    "positive_nmp": len(nmp_markers),
    "negative_pluripotent": len(pluripotent_negative_markers),
    "negative_lpm": len(lpm_negative_markers),
    "negative_nmp": len(nmp_negative_markers),
}))

display_columns = [
    "gene_id", "gene_name", "anova_fdr",
    "mean_log2cpm_pluripotent", "mean_log2cpm_lpm", "mean_log2cpm_nmp",
    "pluri_minus_lpm", "pluri_minus_nmp", "lpm_minus_nmp",
]

print("Top positive markers")
display(pluripotent_markers[display_columns].head(10))
display(lpm_markers[display_columns].head(10))
display(nmp_markers[display_columns].head(10))

print("Top negative markers")
display(pluripotent_negative_markers[display_columns].head(10))
display(lpm_negative_markers[display_columns].head(10))
display(nmp_negative_markers[display_columns].head(10))

print(f"Saved marker statistics to: {marker_results_path}")
print(f"Saved Pluripotent markers to: {pluripotent_markers_path}")
print(f"Saved LPM markers to: {lpm_markers_path}")
print(f"Saved NMP markers to: {nmp_markers_path}")
print(f"Saved low-in-Pluripotent genes to: {pluripotent_negative_markers_path}")
print(f"Saved low-in-LPM genes to: {lpm_negative_markers_path}")
print(f"Saved low-in-NMP genes to: {nmp_negative_markers_path}")

In [ ]:
# Keep text editable in Illustrator by embedding TrueType fonts in the PDF.
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

requested_source = log2_df.copy()
requested_source["gene_name_upper"] = requested_source["gene_name"].fillna("").astype(str).str.upper()
requested_gene_names_upper = [gene.upper() for gene in requested_genes]

requested_heatmap_source = requested_source[requested_source["gene_name_upper"].isin(requested_gene_names_upper)].copy()
requested_heatmap_source["gene_name"] = pd.Categorical(requested_heatmap_source["gene_name"], categories=requested_genes, ordered=True)
requested_heatmap_source = requested_heatmap_source.sort_values("gene_name").drop_duplicates(subset=["gene_name"])

found_genes = requested_heatmap_source["gene_name"].astype(str).tolist()
missing_genes = [gene for gene in requested_genes if gene.upper() not in set(requested_heatmap_source["gene_name_upper"])]
print(f"Found genes: {found_genes}")
print(f"Missing genes: {missing_genes}")

requested_heatmap_df = requested_heatmap_source.set_index("gene_name")[requested_sample_order].copy()
requested_heatmap_z = requested_heatmap_df.apply(row_zscore, axis=1).fillna(0)

g = sns.clustermap(
    requested_heatmap_z,
    cmap="vlag",
    col_cluster=False,
    row_cluster=False,
    xticklabels=True,
    yticklabels=True,
    linewidths=0,
    figsize=(10, max(6, len(requested_heatmap_z) * 0.35)),
    col_colors=column_colors_requested,
)
g.ax_heatmap.set_xlabel("Samples")
g.ax_heatmap.set_ylabel("Gene name")
g.fig.suptitle("Requested Gene List Heatmap", y=1.02)
g.fig.savefig(requested_gene_heatmap_path, dpi=300, bbox_inches="tight")
g.fig.savefig(requested_gene_heatmap_pdf_path, bbox_inches="tight")
plt.show()

print(f"Genes plotted: {len(requested_heatmap_df)}")
print(f"Saved requested gene heatmap PNG to: {requested_gene_heatmap_path}")
print(f"Saved requested gene heatmap PDF to: {requested_gene_heatmap_pdf_path}")

In [ ]:
requested_heatmap_z.to_csv(OUT / "ed10g_heatmap_zscores.tsv", sep="\t")
print(f"ED 10g: wrote {requested_heatmap_z.size} plotted values to bulkseq/output/ed10g_heatmap_zscores.tsv")